# Decoding: beam search with the char-LM and word LM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enph-479-edge-ai-stt/enph-479-edge-ai-stt/blob/main/training/notebooks/decode.ipynb)

Decodes dev-clean with a trained acoustic model (AM) and character model (CM) from earlier runs, plus OpenSLR's pruned trigram as the word LM (WLM), fused by CTC prefix beam search (`training/decode/beam.py`, the paper's N-best search). Tunes the LM weights on a sample of dev-clean, then reports WER and CER on held-out utterances for each piece added in turn: greedy, beam search alone, + CM, + WLM, + both. **Runtime: T4 GPU** (the AM and CM run on it; the search itself is CPU-bound Python). Saves the log and the results to a run folder on Google Drive.

The search is the one the board will run: `runtime/` has a line-for-line copy, and both reproduce `shared/golden/decode/`.

## 1. Pull the repo

In [ ]:
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/enph-479-edge-ai-stt/enph-479-edge-ai-stt.git"
REPO_DIR = "/content/enph-479-edge-ai-stt"
BRANCH = "main"  # set to a feature branch to test it before merging

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
# Always land on the tip of BRANCH, even if an earlier run left the clone on another branch.
git = ["git", "-C", REPO_DIR]
subprocess.run([*git, "fetch", "origin", BRANCH], check=True)
subprocess.run([*git, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
print(subprocess.run([*git, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# Installs this checkout's package and its dependencies (jiwer, soundfile, and kenlm for
# the word LM, which builds from source: about a minute); torch and torchaudio come with Colab.
training_dir = os.path.join(REPO_DIR, "training")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{training_dir}[wlm]"], check=True
)

# The editable install only takes effect at interpreter start, so put src/ on the
# path for this already-running kernel.
sys.path.insert(0, os.path.join(training_dir, "src"))
importlib.invalidate_caches()

## 2. Mount Drive and configure the run

Set `AM_RUN` and `CM_RUN` to the Drive folders the training notebooks saved to. `SUFFIX = "_6bit"` decodes with the 6-bit models, the ones the fabric runs; `""` uses the float ones. The results go to `MyDrive/enph479-stt/runs/<start time>-decode/`.

The weights are tuned on `TUNE_UTTS` random dev-clean utterances with a narrower beam (`TUNE_BEAM`), one decode per grid point, and the results are reported on `EVAL_UTTS` others (`None` for all the rest) at the full beam, the paper's 128. Not yet timed on Colab: the decode prints seconds per utterance every 200 utterances, so shrink the grid or the sizes if it runs long.

In [ ]:
import time

from google.colab import drive

from training.decode.beam import Config

drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/enph479-stt/runs"
AM_RUN = f"{RUNS}/YYYYMMDD-HHMM-am"  # set to the AM run to decode with
CM_RUN = f"{RUNS}/YYYYMMDD-HHMM-cm"  # set to the CM run
SUFFIX = "_6bit"
RUN_DIR = f"{RUNS}/{time.strftime('%Y%m%d-%H%M')}-decode"

TUNE_UTTS = 100
EVAL_UTTS = 500
TUNE_BEAM = 32
GRID = {"alpha": [0.25, 0.5, 1.0], "beta": [0.5, 1.0, 2.0], "gamma": [0.0, 1.0, 2.0]}
cfg = Config(beam=128)
print(RUN_DIR)
cfg

## 3. Models and data

Loads both checkpoints (the AM's carries its CMVN statistics), downloads dev-clean (337 MB) and the word LM (34 MB gzipped ARPA: 200K words, 2.5M bigrams, 1.1M trigrams, from the same LM text as the CM with the dev and test books excluded) to `/content/data`, and splits dev-clean into the tuning and evaluation utterances. The word LM's perplexity and OOV rate on the dev-clean transcripts are its own sanity check, before any decoding.

In [ ]:
import random

from training.am.dataset import list_utterances
from training.data.librispeech import download_subset, download_word_lm
from training.decode import run
from training.wlm.model import KenLM, perplexity

am, cm_model, mean, std = run.load_models(
    f"{AM_RUN}/am{SUFFIX}.pt", f"{CM_RUN}/cm{SUFFIX}.pt", "cuda"
)
cm = run.TorchCharLM(cm_model)
items = list_utterances(download_subset("dev-clean", "/content/data"))
wlm = KenLM(download_word_lm("/content/data"))
refs = [text for _, text in items]

order = random.Random(0).sample(range(len(items)), len(items))
tune_idx = order[:TUNE_UTTS]
eval_idx = order[TUNE_UTTS:][:EVAL_UTTS]
print(f"dev-clean {len(items)} utts: tune on {len(tune_idx)}, evaluate on {len(eval_idx)}")

ppl, oov = perplexity(wlm, refs)
run.log("decode.log", f"word LM on dev-clean: perplexity {ppl:.0f}, OOV rate {oov:.2%}")

## 4. AM output

Runs the AM over all of dev-clean once and keeps its per-frame log-probs, so every decode below reuses them. Greedy decoding (per-frame argmax, CTC collapse) on all of dev-clean should match the training notebook's last test CER.

In [ ]:
logps = run.am_log_probs(am, items, mean, std)
wer, cer = run.error_rates(refs, run.greedy(logps))
run.log("decode.log", f"greedy, all of dev-clean: WER {wer:.4f} CER {cer:.4f}")

## 5. Tune the LM weights

Grid search over `alpha` (CM weight), `beta` (WLM weight) and `gamma` (bonus per word) with both LMs, at `TUNE_BEAM`, for the lowest WER on the tuning utterances. Every point is logged.

In [ ]:
from dataclasses import replace

best = run.tune(
    [logps[i] for i in tune_idx],
    [refs[i] for i in tune_idx],
    replace(cfg, beam=TUNE_BEAM),
    GRID,
    "decode.log",
    cm,
    wlm,
)
cfg = replace(best, beam=cfg.beam)
cfg

## 6. Evaluate

Each piece added in turn, on the evaluation utterances at the full beam, with the tuned weights (the rows with one LM keep that LM's weight from the joint tuning). Greedy to "beam, no LM" is what summing over alignments buys; the rest is what each LM buys.

In [ ]:
eval_logps = [logps[i] for i in eval_idx]
eval_refs = [refs[i] for i in eval_idx]
systems = {
    "greedy": lambda: run.greedy(eval_logps),
    "beam, no LM": lambda: run.decode(eval_logps, cfg),
    "beam + CM": lambda: run.decode(eval_logps, cfg, cm),
    "beam + WLM": lambda: run.decode(eval_logps, cfg, wlm=wlm),
    "beam + CM + WLM": lambda: run.decode(eval_logps, cfg, cm, wlm),
}
results = {}
for name, decode in systems.items():
    t0 = time.perf_counter()
    wer, cer = run.error_rates(eval_refs, decode())
    results[name] = {"wer": wer, "cer": cer}
    secs = time.perf_counter() - t0
    run.log("decode.log", f"{name}: WER {wer:.4f} CER {cer:.4f} ({secs:.0f}s)")

## 7. Save to Drive

Writes the log and `decode.json` (which runs and weights were decoded with, the tuned config and the results table) to the run folder, then flushes Drive.

In [ ]:
import json
import shutil
from dataclasses import asdict
from pathlib import Path

Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
shutil.copy("decode.log", RUN_DIR)
summary = {
    "am": f"{AM_RUN}/am{SUFFIX}.pt",
    "cm": f"{CM_RUN}/cm{SUFFIX}.pt",
    "tune_utts": len(tune_idx),
    "eval_utts": len(eval_idx),
    "config": asdict(cfg),
    "results": results,
}
Path(RUN_DIR, "decode.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"saved the log and decode.json to {RUN_DIR}")
drive.flush_and_unmount()